In [ ]:
# python d_analysis/probe_disentanglement.py --model-num 11 --epoch 399 --metrics factorvae mig


In [2]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import math

notebook_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(notebook_dir))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
print(sys.path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

%load_ext autoreload
%autoreload 2

['/mnt/ceph/users/blyo1/projects/hdiva', '/mnt/sw/nix/store/71ksmx7k6xy3v9ksfkv5mp5kxxp64pd6-python-3.10.13-view/lib/python310.zip', '/mnt/sw/nix/store/71ksmx7k6xy3v9ksfkv5mp5kxxp64pd6-python-3.10.13-view/lib/python3.10', '/mnt/sw/nix/store/71ksmx7k6xy3v9ksfkv5mp5kxxp64pd6-python-3.10.13-view/lib/python3.10/lib-dynload', '', '/mnt/home/blyo1/venvs/py310/lib/python3.10/site-packages']


# model imae 7

In [3]:
from d_analysis.probe_disentanglement import (
    load_encoder, get_dsprites_codes, load_dsprites_labels,
    run_probes, run_factor_vae, run_mig,
    print_probe_report, print_factor_vae_report, print_mig_report, plot_per_dim,
)

project, model_num, epoch = "imae_dsprites", 17, "last"
tag = f"{project}_{model_num}_{epoch}"

encoder, config, ckpt_path = load_encoder("sami", project, model_num, epoch, device=device)
mu, logvar = get_dsprites_codes(
    encoder, ckpt_path, f"{parent_dir}/d_analysis/metrics/codes/{tag}.npz", device=device
)
labels, factor_classes = load_dsprites_labels()


loaded sami encoder from /mnt/ceph/users/blyo1/projects/hdiva/c_training/lightning_checkpoints/imae_dsprites/17-fresh-lion-17-la9zicmz/last.ckpt
cache /mnt/ceph/users/blyo1/projects/hdiva/d_analysis/metrics/codes/imae_dsprites_17_last.npz is from a different checkpoint, re-encoding


encoding: 100%|████████████████████████████████████████████████████████████████████| 1440/1440 [00:05<00:00, 267.87it/s]


cached codes to /mnt/ceph/users/blyo1/projects/hdiva/d_analysis/metrics/codes/imae_dsprites_17_last.npz


In [4]:
config.__dict__

{'strategy': 'ddp',
 'num_nodes': 1,
 'num_gpus_per_node': 3,
 'log_every_n_steps': 10,
 'checkpoint_every_n_epochs': 100,
 'resume_from_checkpoint': False,
 'checkpoint_epoch': 'last',
 'use_pretrained_denoiser_only': True,
 'train_infnet_only': False,
 'pretrained_model_name': 'ddpm_dsprites',
 'pretrained_model_number': 6,
 'pretrained_artifact_id': 'v4',
 'project_dir': '/mnt/home/blyo1/hdiva',
 'model_checkpoint_dir': '/mnt/home/blyo1/hdiva/c_training/lightning_checkpoints',
 'num_epochs': 1200,
 'train_batch_size_per_gpu': 500,
 'lr': 0.001,
 'encoder_lr': 0.001,
 'precision': '32',
 'seed': 43,
 'optimizer': 'adam',
 'latent_dim': 8,
 'num_layers_rec': 3,
 'num_kernels_rec': 32,
 'kernel_size_rec': 3,
 'stride_rec': 2,
 'padding_rec': 1,
 'activation_rec': 'relu',
 'norm_rec': 'gn',
 'num_groups_rec': 8,
 'bias_rec': True,
 'adaptive_avg_pool_output_size': 3,
 'time_embedding_method_rec': 'film',
 'logvar_init_bias': -2.0,
 'res_zero_init': True,
 'num_res_blocks_rec': 2,
 'mode

In [84]:
fvae = run_factor_vae(mu, num_repeats=5)
mig = run_mig(mu, factor_classes, num_repeats=5)
print_factor_vae_report(fvae)
print_mig_report(mig)

fvae["eval_accuracy_mean"], mig["mig_mean"]   # the numbers to report



=== FactorVAE score (5 repeats) ===
eval accuracy 0.429 +- 0.009   (train 0.437; chance 0.20)
active dims (var >= 0.05): [0, 1, 2, 3, 4, 5, 6, 7]
majority-vote assignment (repeat 0): z0->orientation, z1->orientation, z2->pos_x, z3->scale, z4->pos_x, z5->pos_x, z6->pos_y, z7->pos_y

=== MIG (5 repeats, 10000 points, 20 bins) ===
MIG 0.045 +- 0.001
per factor: shape 0.005, scale 0.017, orientation 0.002, pos_x 0.062, pos_y 0.140
MI (nats)        z0     z1     z2     z3     z4     z5     z6     z7
shape          0.01   0.01   0.01   0.01   0.02   0.01   0.02   0.03
scale          0.02   0.03   0.04   0.02   0.11   0.06   0.12   0.15
orientation    0.10   0.05   0.05   0.10   0.04   0.04   0.04   0.04
pos_x          0.04   0.25   0.55   0.04   0.77   0.32   0.21   0.31
pos_y          0.10   0.11   0.08   0.11   0.21   0.32   0.81   0.33


(0.42851999999999996, 0.04506511434055663)

In [4]:
import numpy as np

idx = np.random.default_rng(0).choice(len(mu), 40000, replace=False)
tr, te = idx[:30000], idx[30000:]
probe_results = run_probes(mu[tr], labels[tr], mu[te], labels[te],
                           probes=("linear", "mlp"), per_dim_probe="mlp")
print_probe_report(probe_results)

np.array(probe_results["per_dim"]["matrix"])   # factors x latent dims



=== full-latent probes (test; R^2 or chance-normalized accuracy; 0 = chance, 1 = perfect) ===
factor          linear       mlp
shape            0.127     0.921   acc: linear 0.418, mlp 0.947
scale            0.524     0.922
orientation      0.004     0.478
pos_x            0.700     0.993
pos_y            0.777     0.995
  orientation [linear] by shape: square R2=-0.01 err=22.5deg/90, ellipse R2=0.02 err=39.0deg/180, heart R2=0.00 err=86.6deg/360
  orientation [mlp] by shape: square R2=0.55 err=8.1deg/90, ellipse R2=0.87 err=6.2deg/180, heart R2=0.02 err=73.1deg/360

=== per-dim probes (mlp; one latent dim at a time, test score) ===
factor           z0     z1     z2     z3     z4     z5     z6     z7   best   gap
shape          0.05   0.10   0.05   0.05   0.05   0.05   0.05   0.06   z1     0.04
scale          0.08   0.12   0.08   0.06   0.09   0.07   0.07   0.10   z1     0.01
orientation    0.00   0.01   0.00   0.00  -0.00   0.00   0.00   0.00   z1     0.00
pos_x          0.00   0.24 

array([[ 4.81000000e-02,  1.02100000e-01,  4.87000000e-02,
         5.29000000e-02,  5.18500000e-02,  4.78000000e-02,
         4.75000000e-02,  5.83000000e-02],
       [ 7.99075800e-02,  1.17196677e-01,  7.86571837e-02,
         5.91186814e-02,  8.61062440e-02,  7.41526667e-02,
         7.38299341e-02,  1.03960377e-01],
       [ 1.17966849e-03,  5.12878538e-03,  3.25705995e-03,
         1.66120007e-03, -2.98187718e-04,  1.42449214e-04,
         1.14343172e-03,  4.30018622e-03],
       [ 3.99073311e-03,  2.37466401e-01,  4.03336483e-01,
         2.83739926e-01,  8.73974864e-03,  1.34629188e-01,
         1.18096301e-01,  3.42483057e-01],
       [ 4.08217666e-01,  1.74208382e-02,  1.57568689e-01,
         2.67692726e-01,  3.29268255e-01,  2.96272251e-01,
         3.62774624e-01,  1.09978255e-01]])

# many different models

In [72]:
all_metrics = []

model_nums = [7, 8, 9, 10, 11, 13, 14, 15]
for model_num in model_nums:
    project, model_num, epoch = "imae_dsprites", model_num, "last"
    tag = f"{project}_{model_num}_{epoch}"

    encoder, config, ckpt_path = load_encoder("sami", project, model_num, epoch, device=device)
    mu, logvar = get_dsprites_codes(
        encoder, ckpt_path, f"{parent_dir}/d_analysis/metrics/codes/{tag}.npz", device=device
    )
    labels, factor_classes = load_dsprites_labels()

    fvae = run_factor_vae(mu, num_repeats=5)
    mig = run_mig(mu, factor_classes, num_repeats=5)
    print_factor_vae_report(fvae)
    print_mig_report(mig)

    fvae["eval_accuracy_mean"], mig["mig_mean"]   # the numbers to report

    entry = {
        "project": project,
        "model_num": model_num,
        "epoch": epoch,
        "config": config,
        "factor_vae": fvae,
        "mig": mig,
    }

    all_metrics.append(entry)

loaded sami encoder from /mnt/ceph/users/blyo1/projects/hdiva/c_training/lightning_checkpoints/imae_dsprites/7-zany-shape-7-905h6pt7/last.ckpt
loaded cached codes from /mnt/ceph/users/blyo1/projects/hdiva/d_analysis/metrics/codes/imae_dsprites_7_last.npz

=== FactorVAE score (5 repeats) ===
eval accuracy 0.276 +- 0.005   (train 0.281; chance 0.20)
active dims (var >= 0.05): [0, 1, 2, 3, 4, 5, 6, 7]
majority-vote assignment (repeat 0): z0->pos_y, z1->scale, z2->pos_y, z3->pos_x, z4->pos_y, z5->pos_x, z6->pos_x, z7->scale

=== MIG (5 repeats, 10000 points, 20 bins) ===
MIG 0.006 +- 0.000
per factor: shape 0.003, scale 0.005, orientation 0.002, pos_x 0.012, pos_y 0.006
MI (nats)        z0     z1     z2     z3     z4     z5     z6     z7
shape          0.01   0.02   0.01   0.01   0.01   0.01   0.01   0.01
scale          0.06   0.07   0.06   0.03   0.05   0.05   0.05   0.06
orientation    0.04   0.05   0.04   0.04   0.04   0.04   0.04   0.04
pos_x          0.38   0.23   0.42   0.36   0.36  

encoding: 100%|████████████████████████████████████████████████████████████████████| 1440/1440 [00:04<00:00, 346.36it/s]


cached codes to /mnt/ceph/users/blyo1/projects/hdiva/d_analysis/metrics/codes/imae_dsprites_15_last.npz

=== FactorVAE score (5 repeats) ===
eval accuracy 0.656 +- 0.007   (train 0.657; chance 0.20)
active dims (var >= 0.05): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]
majority-vote assignment (repeat 0): z0->shape, z1->orientation, z2->orientation, z3->orientation, z4->shape, z5->orientation, z6->pos_x, z7->shape, z8->shape, z9->shape, z10->pos_x, z11->pos_y, z12->shape, z13->pos_y, z14->scale, z15->orientation, z16->orientation, z17->orientation, z18->shape, z19->scale, z20->shape, z21->orientation, z22->pos_y, z23->shape, z24->pos_x, z25->pos_y, z26->shape, z27->pos_x, z28->shape, z29->pos_x, z30->shape, z31->pos_y

=== MIG (5 repeats, 10000 points, 20 bins) ===
MIG 0.031 +- 0.002
per factor: shape 0.075, scale 0.044, orientation 0.001, pos_x 0.025, pos_y 0.009
MI (nats)        z0     z1     z2     z3     z4 

In [79]:
for model_num in model_nums:
    metrics = [m for m in all_metrics if m["model_num"] == model_num][0]
    print(f"model_num: {model_num}, factor_vae: {metrics['factor_vae']['eval_accuracy_mean']:.4f}, mig: {metrics['mig']['mig_mean']:.4f}")

model_num: 7, factor_vae: 0.2762, mig: 0.0055
model_num: 8, factor_vae: 0.4520, mig: 0.0451
model_num: 9, factor_vae: 0.4732, mig: 0.0490
model_num: 10, factor_vae: 0.6528, mig: 0.2101
model_num: 11, factor_vae: 0.4841, mig: 0.0563
model_num: 13, factor_vae: 0.6144, mig: 0.0183
model_num: 14, factor_vae: 0.4547, mig: 0.0479
model_num: 15, factor_vae: 0.6563, mig: 0.0310


In [80]:
# show the model_num with the best factor_vae and mig scores
best_factor_vae_model_num = max(all_metrics, key=lambda x: x["factor_vae"]["eval_accuracy_mean"])["model_num"]
best_mig_model_num = max(all_metrics, key=lambda x: x["mig"]["mig_mean"])["model_num"]
print(f"Best factor_vae model_num: {best_factor_vae_model_num}, Best mig model_num: {best_mig_model_num}")

Best factor_vae model_num: 15, Best mig model_num: 10


# compare information via mlp for model 10 (infomax) and model 17 (elbo)

In [ ]:
import numpy as np

idx = np.random.default_rng(0).choice(len(mu), 40000, replace=False)
tr, te = idx[:30000], idx[30000:]
probe_results = run_probes(mu[tr], labels[tr], mu[te], labels[te],
                           probes=("linear", "mlp"), per_dim_probe="mlp")
print_probe_report(probe_results)

np.array(probe_results["per_dim"]["matrix"])   # factors x latent dims
